In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import numpy as np
from generator import make_scenario, price_dp
from env import play, score
from bots import (scripted_dealer_quote, scripted_client_respond,
                  scripted_dealer_decide, random_bots)

G = yaml.safe_load(open("config.yaml"))["calibration"]["g_points"]
print(f"g = {G}")


In [ ]:
#test

def predict(scn, g):
    """Independent prediction of the scripted-bot outcome. Does NOT use play()."""
    p = round(scn.m + g, price_dp)
    price = p if p <= scn.r - 0.5 * g else round(scn.m, price_dp)
    return price, (scn.r - price) / (scn.r - scn.c)

In [ ]:
print(f"{'seed':>5}  {'status':<17}{'play':>9}{'pred':>10}{'cap play':>10}{'cap pred':>10}   viol")
all_ok = True
for s in range(1, 21):
    scn = make_scenario(s, G)
    out = score(scn, play(scn, G, scripted_dealer_quote,
                          scripted_client_respond, scripted_dealer_decide))
    exp_price, exp_cap = predict(scn, G)
    ok = out.price == exp_price and abs(out.client_capture - exp_cap) < 1e-12
    all_ok &= ok
    print(f"{s:>5}  {out.status:<17}{out.price:>9.3f}{exp_price:>10.3f}"
          f"{out.client_capture:>10.4f}{exp_cap:>10.4f}   "
          f"{len(out.violations)}  {'OK' if ok else 'FAIL'}")

print(f"\nGATE A: {'PASS' if all_ok else 'FAIL'}")